# Pulse Algorithm for Feeder Bus Route Optimization

This notebook applies the Pulse Algorithm to the feeder-bus constrained route problem using `INPUT DATA.csv`.

**Editable parameter:** change `TMAX_MINUTES` in Section 1 to test another maximum route duration.

**Main objective:** maximize Total Priority Index (PI) under route time and forward/backward exclusion constraints.

**Main outputs:**
- `pulse_summary.csv`
- `pulse_route_links.csv`
- `pulse_route_nodes_for_google_mymaps.csv`
- `pulse_route_map.kml`
- `pulse_route_map.geojson`
- `pulse_route_map.html`
- `pulse_feeder_bus_outputs.zip`


In [1]:
# ================================================================
# Pulse Algorithm for Feeder Bus Route Optimization
# Case: VNU-HCM first-mile feeder route to Metro Station National University
#
# Research basis:
# - Hemdan et al. (2026): feeder bus design as a constrained route problem
#   using potential demand, priority index, and Pulse Algorithm logic.
# - Lozano & Medaglia (2013): Pulse Algorithm for Constrained Shortest Path.
# - Zhu et al. (2017): circular feeder-bus route structure and potential demand
#   logic based on demand, passenger access distance, and rail-station distance.
#
# Objective:
#   Maximize Total Priority Index (PI)
#
# Main constraints:
#   1. Route starts and ends at START_STATION_ID.
#   2. Total route travel time <= TMAX_MINUTES.
#   3. The final route cannot contain both an original forward demand link
#      and its backward alternative.
#   4. Each original forward demand link can be represented by either its
#      forward or backward direction, but not both.
#   5. Route is treated as a simple circular route: no repeated station
#      except returning to the start station at the final step.
#
# Outputs:
#   1. Total_route_travel_time
#   2. Total PI
#   3. Total Potential Demand
#   4. SAp
#   5. Program Running Time
#   6. Route_map files for Google My Maps
# ================================================================

import os
import math
import time
import json
import heapq
import html
import zipfile
import xml.sax.saxutils as saxutils
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

try:
    import folium
except Exception:
    folium = None




In [2]:
# ================================================================
# 1. USER SETTINGS
# ================================================================
# In Google Colab, upload INPUT DATA.csv to /content, or keep it in your runtime.
INPUT_FILE = "/content/INPUT DATA.csv"

# Local fallback for this generated notebook/script.
LOCAL_FALLBACK_INPUT_FILE = "/mnt/data/INPUT DATA.csv"

# Main route settings.
START_STATION_ID = 31

# Manually change this value when you want another maximum route time.
# Example: 15.0, 18.0, 20.0, 25.0
TMAX_MINUTES = 18

# Search controls.
# For your current network, the Pulse search is usually fast. These limits are
# kept as safeguards for Colab. If the result says SEARCH_LIMIT_REACHED, increase them.
MAX_RUNTIME_SECONDS = 360
MAX_EXPANSIONS = 20_000_000

# Output folder.
OUTPUT_DIR = "/content/pulse_feeder_bus_outputs"
LOCAL_FALLBACK_OUTPUT_DIR = "/mnt/data/pulse_feeder_bus_outputs"

# Earth radius for haversine distance.
EARTH_RADIUS_M = 6_371_000.0




In [3]:
# ================================================================
# 2. PATH HANDLING
# ================================================================
def resolve_input_path():
    """Resolve INPUT DATA.csv in Colab or local execution."""
    if os.path.exists(INPUT_FILE):
        return INPUT_FILE

    if os.path.exists(LOCAL_FALLBACK_INPUT_FILE):
        return LOCAL_FALLBACK_INPUT_FILE

    try:
        from google.colab import files
        print("INPUT DATA.csv was not found. Please upload INPUT DATA.csv.")
        uploaded = files.upload()
        if "INPUT DATA.csv" in uploaded:
            return "/content/INPUT DATA.csv"
    except Exception:
        pass

    raise FileNotFoundError(
        "INPUT DATA.csv was not found. Upload it to Colab or update INPUT_FILE."
    )


def resolve_output_dir():
    """Resolve output directory for Colab or local execution."""
    if os.path.exists("/content"):
        out = Path(OUTPUT_DIR)
    else:
        out = Path(LOCAL_FALLBACK_OUTPUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    return out




In [4]:
# ================================================================
# 3. BASIC GEOSPATIAL FUNCTION
# ================================================================
def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance in meters."""
    phi1 = math.radians(float(lat1))
    phi2 = math.radians(float(lat2))
    dphi = math.radians(float(lat2) - float(lat1))
    dlambda = math.radians(float(lon2) - float(lon1))

    a = (
        math.sin(dphi / 2.0) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0) ** 2
    )
    a = max(0.0, min(1.0, a))
    return 2.0 * EARTH_RADIUS_M * math.asin(math.sqrt(a))




In [5]:
# ================================================================
# 4. DATA PREPARATION: PD, PI, DEMAND KEY, AND SAp BASIS
# ================================================================
def load_and_prepare_input(input_path, start_station_id):
    """
    Prepare the input network.

    Main calculation:
        Potential Demand:
            PD_ij = Pop_ij / S_ij * L_ij

        L_ij:
            straight-line distance from the midpoint of link (i,j)
            to the target metro station.

        PI_ij:
            rank of positive PD_ij among all unique demand links.
            Higher PD receives higher PI.

    Forward/backward rule:
        - Forward link: demand_key = link_name.
        - Backward link: demand_key = forward_source_link_name.
        - U-turn link: PI = 0 and used only as routing connector.
    """
    raw = pd.read_csv(input_path)

    required_columns = [
        "link_name",
        "link_type",
        "from_station_id",
        "to_station_id",
        "from_lat",
        "from_lon",
        "to_lat",
        "to_lon",
        "from_station_name",
        "to_station_name",
        "gis_route_distance_m",
        "average_travel_time_s",
        "assigned_grid_area_km2",
        "assigned_grid_area_m2",
        "Popij",
        "Sij_m",
        "forward_source_link_name",
    ]

    missing = [c for c in required_columns if c not in raw.columns]
    if missing:
        raise ValueError(f"Missing required columns in INPUT DATA.csv: {missing}")

    df = raw.copy()

    # Standardize fields.
    df["link_name"] = df["link_name"].astype(str).str.strip()
    df["link_type"] = df["link_type"].astype(str).str.strip()

    integer_columns = ["from_station_id", "to_station_id"]
    for c in integer_columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype(int)

    numeric_columns = [
        "from_lat",
        "from_lon",
        "to_lat",
        "to_lon",
        "gis_route_distance_m",
        "average_travel_time_s",
        "average_speed_kmh",
        "assigned_grid_area_km2",
        "assigned_grid_area_m2",
        "Popij",
        "Sij_m",
    ]

    for c in numeric_columns:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

    # Build station table.
    station_records = []
    for _, r in df.iterrows():
        station_records.append(
            {
                "station_id": int(r["from_station_id"]),
                "latitude": float(r["from_lat"]),
                "longitude": float(r["from_lon"]),
                "station_name": str(r["from_station_name"]),
            }
        )
        station_records.append(
            {
                "station_id": int(r["to_station_id"]),
                "latitude": float(r["to_lat"]),
                "longitude": float(r["to_lon"]),
                "station_name": str(r["to_station_name"]),
            }
        )

    stations = (
        pd.DataFrame(station_records)
        .drop_duplicates("station_id")
        .sort_values("station_id")
        .reset_index(drop=True)
    )

    if start_station_id not in set(stations["station_id"]):
        raise ValueError(f"START_STATION_ID={start_station_id} is not found in the station table.")

    start_row = stations.loc[stations["station_id"].eq(start_station_id)].iloc[0]
    start_lat = float(start_row["latitude"])
    start_lon = float(start_row["longitude"])

    # Link midpoint and station-distance factor Lij.
    df["mid_lat"] = (df["from_lat"] + df["to_lat"]) / 2.0
    df["mid_lon"] = (df["from_lon"] + df["to_lon"]) / 2.0

    df["Lij_to_station_m"] = [
        haversine_m(mlat, mlon, start_lat, start_lon)
        for mlat, mlon in zip(df["mid_lat"], df["mid_lon"])
    ]

    # Base potential demand per row.
    is_uturn = df["link_type"].str.lower().eq("u-turn")

    valid_pd = (~is_uturn) & (df["Popij"] > 0) & (df["Sij_m"] > 0)
    df["PD_base"] = 0.0
    df.loc[valid_pd, "PD_base"] = (
        df.loc[valid_pd, "Popij"]
        / df.loc[valid_pd, "Sij_m"]
        * df.loc[valid_pd, "Lij_to_station_m"]
    )

    # Demand-key definition.
    df["demand_key"] = df["link_name"]
    df["metric_source_link_name"] = df["link_name"]

    source_col = df["forward_source_link_name"].fillna("").astype(str).str.strip()
    is_backward = df["link_type"].str.lower().eq("backward") & source_col.ne("")

    df.loc[is_backward, "demand_key"] = source_col[is_backward]
    df.loc[is_backward, "metric_source_link_name"] = source_col[is_backward]

    df.loc[is_uturn, "demand_key"] = "UTURN_" + df.loc[is_uturn, "link_name"].astype(str)
    df.loc[is_uturn, "metric_source_link_name"] = df.loc[is_uturn, "link_name"].astype(str)

    # Backward links inherit the source forward-link demand metrics.
    source_df = df.set_index("link_name", drop=False)

    inherited_metrics = []
    for _, r in df.iterrows():
        link_type = str(r["link_type"]).lower()
        source_name = str(r["metric_source_link_name"])

        if source_name in source_df.index:
            source_row = source_df.loc[source_name]
            if isinstance(source_row, pd.DataFrame):
                source_row = source_row.iloc[0]
        else:
            source_row = r

        if link_type == "u-turn":
            inherited_metrics.append(
                {
                    "PDij": 0.0,
                    "Pop_key": 0.0,
                    "Area_key_km2": 0.0,
                    "Area_key_m2": 0.0,
                }
            )
        else:
            inherited_metrics.append(
                {
                    "PDij": float(source_row["PD_base"]),
                    "Pop_key": float(source_row["Popij"]),
                    "Area_key_km2": float(source_row["assigned_grid_area_km2"]),
                    "Area_key_m2": float(source_row["assigned_grid_area_m2"]),
                }
            )

    df = pd.concat([df, pd.DataFrame(inherited_metrics)], axis=1)

    # PI ranking by unique demand key.
    non_uturn_unique = (
        df[~is_uturn]
        .drop_duplicates("demand_key")[["demand_key", "PDij"]]
        .copy()
    )

    positive_keys = (
        non_uturn_unique[non_uturn_unique["PDij"] > 0]
        .sort_values(["PDij", "demand_key"], ascending=[True, True])
        .reset_index(drop=True)
    )

    positive_keys["PIij"] = np.arange(1, len(positive_keys) + 1, dtype=int)

    df = df.merge(positive_keys[["demand_key", "PIij"]], on="demand_key", how="left")
    df["PIij"] = df["PIij"].fillna(0).astype(int)

    # U-turn rows have zero demand contribution.
    is_uturn = df["link_type"].str.lower().eq("u-turn")
    df.loc[is_uturn, ["PDij", "Pop_key", "Area_key_km2", "Area_key_m2", "PIij"]] = 0

    # Edge id.
    df = df.reset_index(drop=True)
    df["edge_id"] = df.index.astype(int)

    return df, stations




In [6]:
# ================================================================
# 5. FEEDER GRAPH CLASS
# ================================================================
class FeederGraph:
    def __init__(self, edges_df, stations_df, start_station_id):
        self.edges_df = edges_df.copy()
        self.stations_df = stations_df.copy()
        self.start = int(start_station_id)

        self.adj = defaultdict(list)
        self.rev = defaultdict(list)
        self.edge_by_id = {}
        self.nodes = set()

        for _, r in self.edges_df.iterrows():
            e = {
                "edge_id": int(r["edge_id"]),
                "link_name": str(r["link_name"]),
                "link_type": str(r["link_type"]),
                "u": int(r["from_station_id"]),
                "v": int(r["to_station_id"]),
                "from_station_name": str(r["from_station_name"]),
                "to_station_name": str(r["to_station_name"]),
                "from_lat": float(r["from_lat"]),
                "from_lon": float(r["from_lon"]),
                "to_lat": float(r["to_lat"]),
                "to_lon": float(r["to_lon"]),
                "time_s": float(r["average_travel_time_s"]),
                "distance_m": float(r["gis_route_distance_m"]),
                "PI": int(r["PIij"]),
                "PD": float(r["PDij"]),
                "Pop": float(r["Pop_key"]),
                "Area_km2": float(r["Area_key_km2"]),
                "Area_m2": float(r["Area_key_m2"]),
                "demand_key": str(r["demand_key"]),
            }

            self.adj[e["u"]].append(e)
            self.rev[e["v"]].append(e)
            self.edge_by_id[e["edge_id"]] = e
            self.nodes.add(e["u"])
            self.nodes.add(e["v"])

        # Explore high-value links earlier. This does not change exactness.
        for u in self.adj:
            self.adj[u].sort(
                key=lambda x: (
                    x["PI"] / max(x["time_s"], 1e-9),
                    x["PI"],
                    -x["time_s"],
                ),
                reverse=True,
            )

        # Unique demand-key totals for fair coverage denominator.
        unique_keys = (
            self.edges_df[self.edges_df["PIij"] > 0]
            .drop_duplicates("demand_key")
            .copy()
        )

        self.total_available_PI = int(unique_keys["PIij"].sum())
        self.total_available_PD = float(unique_keys["PDij"].sum())
        self.total_available_population = float(unique_keys["Pop_key"].sum())
        self.total_available_area_km2 = float(unique_keys["Area_key_km2"].sum())
        self.total_available_area_m2 = float(unique_keys["Area_key_m2"].sum())
        self.n_unique_positive_demand_links = int(unique_keys["demand_key"].nunique())

        self.min_time_to_start = self.compute_min_time_to_start()

    def compute_min_time_to_start(self):
        """
        Reverse Dijkstra from the start station.
        min_time_to_start[n] is the shortest travel time from node n back to start.
        Used for Pulse infeasibility pruning.
        """
        dist = {n: math.inf for n in self.nodes}
        dist[self.start] = 0.0
        pq = [(0.0, self.start)]

        while pq:
            d, node = heapq.heappop(pq)
            if d > dist[node] + 1e-9:
                continue

            for e in self.rev.get(node, []):
                nd = d + e["time_s"]
                if nd < dist[e["u"]] - 1e-9:
                    dist[e["u"]] = nd
                    heapq.heappush(pq, (nd, e["u"]))

        return dist

    def route_nodes(self, route):
        if not route:
            return []
        nodes = [route[0]["u"]]
        nodes.extend(e["v"] for e in route)
        return nodes

    def is_feasible(self, route, tmax_s, enforce_simple_route=True):
        if not route:
            return False, "empty_route"

        if route[0]["u"] != self.start:
            return False, "does_not_start_at_target_station"

        if route[-1]["v"] != self.start:
            return False, "does_not_end_at_target_station"

        elapsed = 0.0
        previous_node = self.start
        used_edges = set()
        used_demand_keys = set()
        visited_nodes = {self.start}

        for idx, e in enumerate(route):
            if e["u"] != previous_node:
                return False, "route_not_continuous"

            elapsed += e["time_s"]
            if elapsed > tmax_s + 1e-9:
                return False, "travel_time_exceeds_Tmax"

            if e["edge_id"] in used_edges:
                return False, "repeated_directed_edge"
            used_edges.add(e["edge_id"])

            if e["PI"] > 0:
                if e["demand_key"] in used_demand_keys:
                    return False, "forward_backward_or_duplicate_demand_key"
                used_demand_keys.add(e["demand_key"])

            next_node = e["v"]

            if enforce_simple_route:
                if next_node == self.start:
                    if idx != len(route) - 1:
                        return False, "returned_to_start_before_final_step"
                else:
                    if next_node in visited_nodes:
                        return False, "repeated_station"
                    visited_nodes.add(next_node)

            previous_node = next_node

        return True, "OK"

    def metrics(self, route):
        used_keys = set()

        total_pi = 0
        total_pd = 0.0
        total_pop = 0.0
        total_area_km2 = 0.0
        total_area_m2 = 0.0

        for e in route:
            if e["PI"] <= 0:
                continue

            key = e["demand_key"]
            if key in used_keys:
                continue

            used_keys.add(key)
            total_pi += e["PI"]
            total_pd += e["PD"]
            total_pop += e["Pop"]
            total_area_km2 += e["Area_km2"]
            total_area_m2 += e["Area_m2"]

        total_time_s = sum(e["time_s"] for e in route)
        total_distance_m = sum(e["distance_m"] for e in route)

        sap_percent = (
            total_area_km2 / self.total_available_area_km2 * 100.0
            if self.total_available_area_km2 > 0
            else 0.0
        )

        return {
            "Total_route_travel_time_s": total_time_s,
            "Total_route_travel_time_min": total_time_s / 60.0,
            "Total_route_distance_m": total_distance_m,
            "Total_PI": int(total_pi),
            "Total_Potential_Demand": float(total_pd),
            "Total_Served_Population": float(total_pop),
            "SAp_km2": float(total_area_km2),
            "SAp_m2": float(total_area_m2),
            "SAp_percent": float(sap_percent),
            "Num_route_links": len(route),
            "Num_nodes_including_return": len(self.route_nodes(route)),
            "Route_node_sequence": " -> ".join(map(str, self.route_nodes(route))),
            "Route_link_sequence": " | ".join(e["link_name"] for e in route),
        }




In [7]:
# ================================================================
# 6. PULSE ALGORITHM
# ================================================================
class PulseSolver:
    """
    Pulse-style exact search for the constrained route problem.

    The classical CSP minimizes path cost under a resource constraint.
    Here, the problem is adapted as:
        maximize Total PI
        subject to total travel time <= Tmax

    Pulse components implemented:
        - DFS pulse propagation from the start node.
        - Cycle pruning for simple circular route construction.
        - Resource infeasibility pruning using travel time.
        - Return-feasibility pruning using shortest time back to start.
        - Bound pruning using a fractional upper bound on remaining PI.
        - Forward/backward exclusion through demand_key memory.
    """

    def __init__(
        self,
        graph,
        tmax_minutes,
        max_runtime_seconds=120.0,
        max_expansions=5_000_000,
    ):
        self.graph = graph
        self.tmax_s = float(tmax_minutes) * 60.0
        self.max_runtime_seconds = float(max_runtime_seconds)
        self.max_expansions = int(max_expansions)

        self.best_route = None
        self.best_metrics = None
        self.best_pi = -1
        self.best_pd = -1.0
        self.best_sap = -1.0
        self.best_time_s = math.inf

        self.expansions = 0
        self.prunes = {
            "time_limit": 0,
            "max_expansions": 0,
            "time_resource": 0,
            "return_infeasible": 0,
            "upper_bound": 0,
            "repeated_station": 0,
            "repeated_edge": 0,
            "duplicate_demand_key": 0,
        }

        self.limit_reached = False
        self.start_time = None
        self.upper_bound_items = self.build_upper_bound_items()

    def build_upper_bound_items(self):
        """
        Build an optimistic fractional-knapsack upper bound.

        One item represents one positive unique demand key.
        Item value = PI
        Item time = minimum travel time of any directed representation of that key.

        The bound ignores connectivity and therefore overestimates possible remaining PI.
        That makes it safe for pruning.
        """
        key_best = {}

        for e in self.graph.edge_by_id.values():
            if e["PI"] <= 0:
                continue

            key = e["demand_key"]
            if key not in key_best or e["time_s"] < key_best[key]["time_s"]:
                key_best[key] = {
                    "demand_key": key,
                    "PI": e["PI"],
                    "time_s": max(e["time_s"], 1e-9),
                }

        items = list(key_best.values())
        items.sort(key=lambda x: x["PI"] / x["time_s"], reverse=True)
        return items

    def fractional_pi_upper_bound(self, remaining_time_s, used_demand_keys):
        """
        Optimistic upper bound on extra PI that can still be collected.

        The bound is intentionally optimistic because it ignores route continuity.
        It is only used to safely prune branches that cannot beat the incumbent.
        """
        if remaining_time_s <= 0:
            return 0.0

        ub = 0.0
        rt = float(remaining_time_s)

        for item in self.upper_bound_items:
            if item["demand_key"] in used_demand_keys:
                continue

            if item["time_s"] <= rt:
                ub += item["PI"]
                rt -= item["time_s"]
            else:
                ub += item["PI"] * (rt / item["time_s"])
                break

        return ub

    def incumbent_is_better(self, candidate_route):
        """
        Main ranking:
            1. Higher Total PI
            2. Higher Total Potential Demand
            3. Higher SAp
            4. Lower travel time
        """
        m = self.graph.metrics(candidate_route)

        cand_tuple = (
            m["Total_PI"],
            m["Total_Potential_Demand"],
            m["SAp_percent"],
            -m["Total_route_travel_time_s"],
        )

        best_tuple = (
            self.best_pi,
            self.best_pd,
            self.best_sap,
            -self.best_time_s,
        )

        return cand_tuple > best_tuple

    def update_best(self, route):
        ok, reason = self.graph.is_feasible(route, self.tmax_s)
        if not ok:
            return

        if self.incumbent_is_better(route):
            self.best_route = list(route)
            self.best_metrics = self.graph.metrics(route)
            self.best_pi = self.best_metrics["Total_PI"]
            self.best_pd = self.best_metrics["Total_Potential_Demand"]
            self.best_sap = self.best_metrics["SAp_percent"]
            self.best_time_s = self.best_metrics["Total_route_travel_time_s"]

    def check_global_limits(self):
        if self.expansions >= self.max_expansions:
            self.prunes["max_expansions"] += 1
            self.limit_reached = True
            return True

        if time.perf_counter() - self.start_time > self.max_runtime_seconds:
            self.prunes["time_limit"] += 1
            self.limit_reached = True
            return True

        return False

    def pulse(self, node, elapsed_s, current_pi, path, visited_nodes, used_demand_keys, used_edge_ids):
        """
        Recursive pulse propagation.
        """
        self.expansions += 1

        if self.check_global_limits():
            return

        # Resource and return-feasibility pruning.
        if elapsed_s > self.tmax_s + 1e-9:
            self.prunes["time_resource"] += 1
            return

        min_return = self.graph.min_time_to_start.get(node, math.inf)
        if elapsed_s + min_return > self.tmax_s + 1e-9:
            self.prunes["return_infeasible"] += 1
            return

        # Bound pruning: cannot beat the best incumbent PI.
        remaining_time = self.tmax_s - elapsed_s
        optimistic_pi = current_pi + self.fractional_pi_upper_bound(
            remaining_time, used_demand_keys
        )

        if optimistic_pi < self.best_pi - 1e-9:
            self.prunes["upper_bound"] += 1
            return

        # Propagate through outgoing links.
        for e in self.graph.adj.get(node, []):
            edge_id = e["edge_id"]
            next_node = e["v"]
            new_elapsed = elapsed_s + e["time_s"]

            if edge_id in used_edge_ids:
                self.prunes["repeated_edge"] += 1
                continue

            if new_elapsed > self.tmax_s + 1e-9:
                self.prunes["time_resource"] += 1
                continue

            if e["PI"] > 0 and e["demand_key"] in used_demand_keys:
                self.prunes["duplicate_demand_key"] += 1
                continue

            # Return to start: only allowed as final route closure.
            if next_node == self.graph.start:
                if len(path) == 0:
                    continue

                candidate = path + [e]
                self.update_best(candidate)
                continue

            # Simple route rule: no repeated station before returning to start.
            if next_node in visited_nodes:
                self.prunes["repeated_station"] += 1
                continue

            # Check if next node can still return to start in remaining time.
            if new_elapsed + self.graph.min_time_to_start.get(next_node, math.inf) > self.tmax_s + 1e-9:
                self.prunes["return_infeasible"] += 1
                continue

            new_used_keys = set(used_demand_keys)
            if e["PI"] > 0:
                new_used_keys.add(e["demand_key"])

            # Pre-branch upper bound pruning.
            next_pi = current_pi + e["PI"]
            next_remaining = self.tmax_s - new_elapsed
            optimistic_next = next_pi + self.fractional_pi_upper_bound(
                next_remaining, new_used_keys
            )

            if optimistic_next < self.best_pi - 1e-9:
                self.prunes["upper_bound"] += 1
                continue

            visited_nodes.add(next_node)
            used_edge_ids.add(edge_id)

            self.pulse(
                next_node,
                new_elapsed,
                next_pi,
                path + [e],
                visited_nodes,
                new_used_keys,
                used_edge_ids,
            )

            used_edge_ids.remove(edge_id)
            visited_nodes.remove(next_node)

    def solve(self):
        self.start_time = time.perf_counter()

        self.pulse(
            node=self.graph.start,
            elapsed_s=0.0,
            current_pi=0,
            path=[],
            visited_nodes={self.graph.start},
            used_demand_keys=set(),
            used_edge_ids=set(),
        )

        runtime = time.perf_counter() - self.start_time

        if self.best_route is None:
            status = "NO_FEASIBLE_ROUTE_FOUND"
        elif self.limit_reached:
            status = "SEARCH_LIMIT_REACHED_BEST_FEASIBLE_RETURNED"
        else:
            status = "CERTIFIED_EXACT_WITHIN_SEARCH_SPACE"

        diagnostics = {
            "status": status,
            "runtime_s": runtime,
            "expansions": self.expansions,
            "prunes": self.prunes,
            "TMAX_minutes": self.tmax_s / 60.0,
            "max_runtime_seconds": self.max_runtime_seconds,
            "max_expansions": self.max_expansions,
        }

        return self.best_route, diagnostics




In [8]:
# ================================================================
# 7. EXPORT FUNCTIONS
# ================================================================
def route_links_dataframe(graph, route):
    rows = []
    cumulative_time_s = 0.0

    for seq, e in enumerate(route, start=1):
        cumulative_time_s += e["time_s"]

        rows.append(
            {
                "sequence": seq,
                "link_name": e["link_name"],
                "link_type": e["link_type"],
                "from_station_id": e["u"],
                "to_station_id": e["v"],
                "from_station_name": e["from_station_name"],
                "to_station_name": e["to_station_name"],
                "from_lat": e["from_lat"],
                "from_lon": e["from_lon"],
                "to_lat": e["to_lat"],
                "to_lon": e["to_lon"],
                "travel_time_s": e["time_s"],
                "travel_time_min": e["time_s"] / 60.0,
                "cumulative_time_s": cumulative_time_s,
                "cumulative_time_min": cumulative_time_s / 60.0,
                "distance_m": e["distance_m"],
                "PI": e["PI"],
                "Potential_Demand": e["PD"],
                "Served_Population": e["Pop"],
                "Served_Area_km2": e["Area_km2"],
                "Served_Area_m2": e["Area_m2"],
                "demand_key": e["demand_key"],
            }
        )

    return pd.DataFrame(rows)


def route_nodes_dataframe(graph, route):
    station_dict = graph.stations_df.set_index("station_id").to_dict("index")
    rows = []

    for seq, node in enumerate(graph.route_nodes(route), start=1):
        s = station_dict.get(node, {})
        rows.append(
            {
                "sequence": seq,
                "station_id": node,
                "station_name": s.get("station_name", ""),
                "latitude": s.get("latitude", np.nan),
                "longitude": s.get("longitude", np.nan),
            }
        )

    return pd.DataFrame(rows)


def route_coordinates(route):
    if not route:
        return []

    coords = [(route[0]["from_lon"], route[0]["from_lat"], 0.0)]
    for e in route:
        coords.append((e["to_lon"], e["to_lat"], 0.0))
    return coords


def export_kml(route, output_path):
    coords = route_coordinates(route)
    coord_text = " ".join(f"{lon},{lat},{alt}" for lon, lat, alt in coords)

    placemarks = []

    # Route line.
    placemarks.append(
        f"""
        <Placemark>
            <name>Pulse optimal feeder bus route</name>
            <Style>
                <LineStyle>
                    <color>ff0000ff</color>
                    <width>4</width>
                </LineStyle>
            </Style>
            <LineString>
                <tessellate>1</tessellate>
                <coordinates>{coord_text}</coordinates>
            </LineString>
        </Placemark>
        """
    )

    # Link markers.
    for seq, e in enumerate(route, start=1):
        name = saxutils.escape(f"{seq}. {e['link_name']} ({e['u']}->{e['v']})")
        desc = saxutils.escape(
            f"Type: {e['link_type']} | PI: {e['PI']} | "
            f"PD: {e['PD']:.3f} | Time: {e['time_s']/60:.3f} min | "
            f"Demand key: {e['demand_key']}"
        )

        placemarks.append(
            f"""
            <Placemark>
                <name>{name}</name>
                <description>{desc}</description>
                <Point>
                    <coordinates>{e['to_lon']},{e['to_lat']},0</coordinates>
                </Point>
            </Placemark>
            """
        )

    kml = f"""<?xml version="1.0" encoding="UTF-8"?>
    <kml xmlns="http://www.opengis.net/kml/2.2">
    <Document>
        <name>Pulse Feeder Bus Route</name>
        {''.join(placemarks)}
    </Document>
    </kml>
    """

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(kml)


def export_geojson(route, output_path):
    coords = [[lon, lat] for lon, lat, _ in route_coordinates(route)]

    features = [
        {
            "type": "Feature",
            "properties": {
                "name": "Pulse optimal feeder bus route",
                "model": "Pulse Algorithm",
            },
            "geometry": {
                "type": "LineString",
                "coordinates": coords,
            },
        }
    ]

    for seq, e in enumerate(route, start=1):
        features.append(
            {
                "type": "Feature",
                "properties": {
                    "sequence": seq,
                    "link_name": e["link_name"],
                    "link_type": e["link_type"],
                    "from_station_id": e["u"],
                    "to_station_id": e["v"],
                    "PI": e["PI"],
                    "Potential_Demand": e["PD"],
                    "travel_time_min": e["time_s"] / 60.0,
                    "demand_key": e["demand_key"],
                },
                "geometry": {
                    "type": "Point",
                    "coordinates": [e["to_lon"], e["to_lat"]],
                },
            }
        )

    geojson = {
        "type": "FeatureCollection",
        "features": features,
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(geojson, f, ensure_ascii=False, indent=2)


def export_html_map(route, output_path):
    if folium is None:
        print("folium is not installed. HTML map is skipped.")
        return

    points = [(route[0]["from_lat"], route[0]["from_lon"])]
    points.extend((e["to_lat"], e["to_lon"]) for e in route)

    center_lat = float(np.mean([p[0] for p in points]))
    center_lon = float(np.mean([p[1] for p in points]))

    fmap = folium.Map(location=[center_lat, center_lon], zoom_start=15)

    folium.PolyLine(
        points,
        tooltip="Pulse optimal feeder bus route",
        weight=5,
    ).add_to(fmap)

    for seq, e in enumerate(route, start=1):
        popup = html.escape(
            f"Seq: {seq}<br>"
            f"Link: {e['link_name']}<br>"
            f"Type: {e['link_type']}<br>"
            f"From: {e['u']} - {e['from_station_name']}<br>"
            f"To: {e['v']} - {e['to_station_name']}<br>"
            f"PI: {e['PI']}<br>"
            f"PD: {e['PD']:.3f}<br>"
            f"Time: {e['time_s']/60:.3f} min<br>"
            f"Demand key: {e['demand_key']}"
        )

        folium.CircleMarker(
            location=[e["to_lat"], e["to_lon"]],
            radius=5,
            tooltip=f"{seq}. {e['link_name']}",
            popup=popup,
            fill=True,
        ).add_to(fmap)

    fmap.save(output_path)


def write_outputs(graph, route, diagnostics, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    metrics = graph.metrics(route)

    summary = {
        "Model": "Pulse Algorithm",
        "Objective": "Maximize Total PI",
        "Start_End_Station_ID": graph.start,
        "TMAX_minutes": diagnostics["TMAX_minutes"],
        "Optimality_Status": diagnostics["status"],
        **metrics,
        "Program_Running_Time_s": diagnostics["runtime_s"],
        "Pulse_Expansions": diagnostics["expansions"],
        "Total_available_PI": graph.total_available_PI,
        "Total_available_PD": graph.total_available_PD,
        "Total_available_population": graph.total_available_population,
        "Total_available_area_km2": graph.total_available_area_km2,
        "Total_available_area_m2": graph.total_available_area_m2,
        "Unique_positive_demand_links": graph.n_unique_positive_demand_links,
    }

    summary_df = pd.DataFrame([summary])
    links_df = route_links_dataframe(graph, route)
    nodes_df = route_nodes_dataframe(graph, route)

    summary_path = output_dir / "pulse_summary.csv"
    links_path = output_dir / "pulse_route_links.csv"
    nodes_path = output_dir / "pulse_route_nodes_for_google_mymaps.csv"
    kml_path = output_dir / "pulse_route_map.kml"
    geojson_path = output_dir / "pulse_route_map.geojson"
    html_path = output_dir / "pulse_route_map.html"
    diagnostics_path = output_dir / "pulse_search_diagnostics.json"

    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    links_df.to_csv(links_path, index=False, encoding="utf-8-sig")
    nodes_df.to_csv(nodes_path, index=False, encoding="utf-8-sig")

    export_kml(route, kml_path)
    export_geojson(route, geojson_path)
    export_html_map(route, html_path)

    with open(diagnostics_path, "w", encoding="utf-8") as f:
        json.dump(diagnostics, f, ensure_ascii=False, indent=2)

    # Zip all output files.
    zip_path = output_dir / "pulse_feeder_bus_outputs.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in [
            summary_path,
            links_path,
            nodes_path,
            kml_path,
            geojson_path,
            html_path,
            diagnostics_path,
        ]:
            if file_path.exists():
                zf.write(file_path, arcname=file_path.name)

    return {
        "summary": summary_path,
        "route_links": links_path,
        "route_nodes": nodes_path,
        "kml": kml_path,
        "geojson": geojson_path,
        "html": html_path,
        "diagnostics": diagnostics_path,
        "zip": zip_path,
    }, summary_df, links_df, nodes_df




In [9]:
# ================================================================
# 8. MAIN EXECUTION
# ================================================================
def main():
    input_path = resolve_input_path()
    output_dir = resolve_output_dir()

    print("=" * 72)
    print("PULSE ALGORITHM FOR FEEDER BUS ROUTE OPTIMIZATION")
    print("=" * 72)
    print(f"Input file: {input_path}")
    print(f"Output directory: {output_dir}")
    print(f"Start/end station: {START_STATION_ID}")
    print(f"TMAX_MINUTES: {TMAX_MINUTES}")
    print()

    prepared_edges, stations = load_and_prepare_input(input_path, START_STATION_ID)
    graph = FeederGraph(prepared_edges, stations, START_STATION_ID)

    print("Input data summary")
    print("-" * 72)
    print(f"Number of directed links: {len(prepared_edges):,}")
    print(f"Number of stations: {stations['station_id'].nunique():,}")
    print(f"Unique positive demand links: {graph.n_unique_positive_demand_links:,}")
    print(f"Total available PI: {graph.total_available_PI:,.0f}")
    print(f"Total available PD: {graph.total_available_PD:,.3f}")
    print(f"Total available assigned area: {graph.total_available_area_km2:,.6f} km²")
    print()

    solver = PulseSolver(
        graph=graph,
        tmax_minutes=TMAX_MINUTES,
        max_runtime_seconds=MAX_RUNTIME_SECONDS,
        max_expansions=MAX_EXPANSIONS,
    )

    route, diagnostics = solver.solve()

    if route is None:
        print("No feasible route was found. Try increasing TMAX_MINUTES.")
        return None

    paths, summary_df, links_df, nodes_df = write_outputs(
        graph=graph,
        route=route,
        diagnostics=diagnostics,
        output_dir=output_dir,
    )

    print("Pulse route result")
    print("-" * 72)
    display_cols = [
        "Optimality_Status",
        "Total_route_travel_time_min",
        "Total_PI",
        "Total_Potential_Demand",
        "SAp_percent",
        "Program_Running_Time_s",
        "Route_node_sequence",
    ]

    print(summary_df[display_cols].T)
    print()

    print("Generated files")
    print("-" * 72)
    for key, value in paths.items():
        print(f"{key}: {value}")

    print()
    print("Route links")
    print("-" * 72)
    try:
        from IPython.display import display
        display(links_df)
    except Exception:
        print(links_df)

    return {
        "graph": graph,
        "route": route,
        "diagnostics": diagnostics,
        "summary": summary_df,
        "route_links": links_df,
        "route_nodes": nodes_df,
        "paths": paths,
    }


if __name__ == "__main__":
    RESULTS = main()


PULSE ALGORITHM FOR FEEDER BUS ROUTE OPTIMIZATION
Input file: /content/INPUT DATA.csv
Output directory: /content/pulse_feeder_bus_outputs
Start/end station: 31
TMAX_MINUTES: 18

Input data summary
------------------------------------------------------------------------
Number of directed links: 271
Number of stations: 92
Unique positive demand links: 163
Total available PI: 13,366
Total available PD: 2,929,545.280
Total available assigned area: 12.977950 km²

Pulse route result
------------------------------------------------------------------------
                                                                             0
Optimality_Status                          CERTIFIED_EXACT_WITHIN_SEARCH_SPACE
Total_route_travel_time_min                                          17.559155
Total_PI                                                                   749
Total_Potential_Demand                                            59720.089678
SAp_percent                                      

,sequence,link_name,link_type,from_station_id,to_station_id,from_station_name,to_station_name,from_lat,from_lon,to_lat,...,travel_time_min,cumulative_time_s,cumulative_time_min,distance_m,PI,Potential_Demand,Served_Population,Served_Area_km2,Served_Area_m2,demand_key
0,1,31_30,Forward,31,30,VNU-HCM Metro Station - Opposite VNU-HCM Metro...,Thao Nguyen - 35B National Highway 1 - Linh Tr...,10.866216,106.800528,10.865477,...,2.016667,121.000000,2.016667,874.0,36,1927.395392,704.471103,0.39098,390980.0,31_30
1,2,30_28,Backward,30,28,Thao Nguyen - 35B National Highway 1 - Linh Tr...,VNU-HCM Information Technology Park - Vo Truon...,10.865477,106.793751,10.866788,...,1.200000,193.000000,3.216667,417.0,8,417.138964,87.090783,0.05031,50310.0,28_30
2,3,28_27,U-turn,28,27,VNU-HCM Information Technology Park - Vo Truon...,VNU-HCM Central Library - Vo Truong Toan Stree...,10.866788,106.794586,10.867721,...,0.233333,207.000000,3.450000,116.0,0,0.000000,0.000000,0.00000,0.0,UTURN_28_27
3,4,27_23,Forward,27,23,VNU-HCM Central Library - Vo Truong Toan Stree...,VNU-HCM Central Library - Rear Gate of VNU-HCM...,10.867721,106.795079,10.870383,...,0.850000,258.000000,4.300000,361.0,25,1601.963475,171.415122,0.03780,37800.0,27_23
4,5,23_20,Backward,23,20,VNU-HCM Central Library - Rear Gate of VNU-HCM...,Fishing Lake - Opposite 6/21 Alexandre de Rhod...,10.870383,106.795858,10.871685,...,0.577141,292.628479,4.877141,356.0,69,3853.533204,1547.341477,0.29988,299880.0,21_22
5,6,20_24,Forward,20,24,Fishing Lake - Opposite 6/21 Alexandre de Rhod...,"University of Social Sciences and Humanities, ...",10.871685,106.798147,10.873422,...,1.816667,401.628479,6.693808,537.0,80,5328.713423,557.426369,0.08001,80010.0,20_24
6,7,24_7,Backward,24,7,"University of Social Sciences and Humanities, ...","University of Science, VNU-HCM - Opposite CNG ...",10.873422,106.802275,10.874651,...,1.002713,461.791233,7.696521,536.0,52,2747.488832,15.363263,0.00504,5040.0,5_25
7,8,7_9,Forward,7,9,"University of Science, VNU-HCM - Opposite CNG ...",Area A Bus Terminal - No. 28-30 Nguyen Binh Kh...,10.874651,106.806137,10.875903,...,0.716667,504.791233,8.413187,233.0,62,3415.362865,28.452215,0.00756,7560.0,7_9
8,9,9_16,Forward,9,16,Area A Bus Terminal - No. 28-30 Nguyen Binh Kh...,"AH4 Building, Area A Dormitory - No. 136-138 N...",10.875903,106.807557,10.880700,...,1.166667,574.791233,9.579854,637.0,116,13105.557001,84.903901,0.01638,16380.0,9_16
9,10,16_35,Forward,16,35,"AH4 Building, Area A Dormitory - No. 136-138 N...",Opposite New Eastern Bus Station,10.880700,106.810780,10.880974,...,3.145968,763.549286,12.725821,944.0,103,8476.390617,10.942895,0.00315,3150.0,16_35
